# ✂️ IEEE-CIS Fraud Detection — Data Splitting & Balancing

> **Goal:** Take the processed data, split it into Train/Test sets, and save them as simple, merged CSV files.

### Outputs:
- `data/train_unbalanced.csv` (80% split)
- `data/train_balanced.csv`   (80% split + SMOTE)
- `data/test.csv`             (20% split - Original distribution)

In [1]:
import os
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from imblearn.over_sampling import SMOTE
from imblearn.under_sampling import RandomUnderSampler

DATA_DIR = os.path.abspath('../data')
INPUT_CSV = os.path.join(DATA_DIR, 'processed_train.csv')

print('Loading processed data...')
df = pd.read_csv(INPUT_CSV)
print(f'Loaded {df.shape[0]:,} rows.')

Loading processed data...


Loaded 590,540 rows.


## 1 — Train/Test Split

In [2]:
y = df['isFraud']
X = df.drop(columns=['isFraud'], errors='ignore') # Keep TransactionID for identification if needed, but usually we drop it later

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f'Train set: {X_train.shape} (Fraud: {y_train.sum()} / {len(y_train)})')
print(f'Test set : {X_test.shape} (Fraud: {y_test.sum()} / {len(y_test)})')

# Merge and Save Unbalanced splits
train_unbal = X_train.copy()
train_unbal['isFraud'] = y_train
train_unbal.to_csv(os.path.join(DATA_DIR, 'train_unbalanced.csv'), index=False)

test_merged = X_test.copy()
test_merged['isFraud'] = y_test
test_merged.to_csv(os.path.join(DATA_DIR, 'test.csv'), index=False)

print('✅ Unbalanced train and test files saved.')

Train set: (472432, 227) (Fraud: 16530 / 472432)
Test set : (118108, 227) (Fraud: 4133 / 118108)


✅ Unbalanced train and test files saved.


## 2 — Apply SMOTE & Save Balanced Train

In [3]:
print('Applying SMOTE to Training set...')
smote = SMOTE(sampling_strategy=0.2, random_state=42)
# Drop TransactionID before SMOTE as it's not a feature
X_train_clean = X_train.drop(columns=['TransactionID'], errors='ignore')
X_train_res, y_train_res = smote.fit_resample(X_train_clean, y_train)

print('Applying Undersampling to reach 1:1...')
rus = RandomUnderSampler(sampling_strategy=0.5, random_state=42)
X_train_bal, y_train_bal = rus.fit_resample(X_train_res, y_train_res)

print(f'Final Balanced Train: {X_train_bal.shape}')
print(pd.Series(y_train_bal).value_counts())

# Merge and Save Balanced split
train_bal = pd.DataFrame(X_train_bal, columns=X_train_clean.columns)
train_bal['isFraud'] = y_train_bal
train_bal.to_csv(os.path.join(DATA_DIR, 'train_balanced.csv'), index=False)

print('✅ Balanced training file saved.')

Applying SMOTE to Training set...


Applying Undersampling to reach 1:1...


Final Balanced Train: (273540, 226)
isFraud
0    182360
1     91180
Name: count, dtype: int64


✅ Balanced training file saved.
